# Step 12: Spatial Microenvironment Figures and Analysis

This notebook generates publication-ready figures to characterize spatial cell distributions within tumor and normal epithelia microenvironments.

## Overview

Analyzes spatial relationships using computed distance metrics to investigate:

- **Cell clustering patterns**: How specific cell types distribute relative to tissue boundaries
- **Microenvironment composition**: Phenotypic diversity at different distances from regions
- **Distance gradients**: Changes in cell populations as distance increases from boundaries
- **Patient-level and batch-level variation**: Patterns across timepoints and treatment arms

## Input
- Unified CSV from Step 11 (with tumor_bed, distance_tumor, distance_epithelia, bed_roi columns)
- Distance columns: signed distances with negative values inside regions, positive outside

## Output
- Publication-quality PNG figures (300 dpi)
- Multiple analysis types: distributions, heatmaps, violin plots, stacked area charts
- Summary statistics and tables

## Key Features
- **Flexible distance column selection**: Analyze any distance metric (distance_tumor, distance_epithelia, etc.)
- **Multi-level grouping**: Separate analyses by patient, timepoint, arm, response status
- **Large dataset handling**: 44 samples, 242 tissue_roi, 2.6M+ cells
- **Statistical analysis**: Significance testing, permutation analysis
- **Publication-ready**: Professional styling, clear legends, publication fonts

## Imports and Setup

In [ ]:
# Imports & Environment Setup
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from scipy import stats
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
import time
import itertools
from tqdm import tqdm
from shapely.geometry import Point, Polygon, MultiPoint, mapping
from shapely.ops import unary_union
import json
import uuid

warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.utils import log_time

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"Project root: {project_root}")
print("Step 12: Spatial Figures and Analysis Setup Complete")

## Configuration

Configure analysis parameters including input/output paths, distance metrics to analyze, grouping variables, and figure styling options.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# I/O Paths
INPUT_DIR = project_root / 'outputs' / 'phenotyping'
OUTPUT_DIR = project_root / 'outputs' / 'spatial_figures'
#OUTPUT_DIR = "Y:\\Publications\\SPACEMAP_V9\\outputs\\phenotyping\\spatial_figures"

# Input file (unified phenotyping output from Step 11)
UNIFIED_INPUT_FILE = INPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== DISTANCE METRIC SELECTION =====
# Choose which distance column to analyze
DISTANCE_COLUMN = 'distance_tumor'  # Options: 'distance_tumor', 'distance_epithelia'
DISTANCE_BED = 'tumor'  # Corresponding bed: 'tumor' (1) or 'epithelia' (0)

# ===== CELL TYPE / PHENOTYPE SELECTION =====
# Column to analyze (phenotype to study across distance)
PHENOTYPE_COLUMN = 'phenotype_subsets'  # e.g., 'phenotype_subsets', 'phenotype_ml_mr', 'cell_type'

# Specific phenotypes to highlight (None = all)
PHENOTYPES_OF_INTEREST = None  # e.g., ['T cell', 'Macrophage'] or None for all

# ===== GROUPING / FILTERING =====
# Patient selection
PATIENT_SELECTION = 'all'  # 'all' or list like [1, 5, 9]
PATIENT_GROUPING = 'merged'  # 'merged' (pool all) or 'split' (separate plots per patient)

# Comparison variables for stratification
COMPARISON_VARS = ['arm', 'response', 'timepoint']  # Variables to stratify by
# Available levels for each (will be auto-detected from data)
COMPARISON_LEVELS = None  # Set to None to auto-detect

# Fixed filters (None = no filter)
FIXED_ARM = None  # e.g., 1 or None
FIXED_RESPONSE = None  # e.g., 'PR' or None
FIXED_TIMEPOINT = None  # e.g., 1 or None
FIXED_PHENOTYPE_FILTER = None  # Filter to specific phenotype_ml_mr (e.g., 'Immune')

# ===== FIGURE STYLING =====
FIGURE_DPI = 300  # Resolution for saved figures
FIGURE_FORMAT = 'png'  # Format: 'png', 'pdf', 'svg'

# Distance binning (for histograms and distributions)
NUM_BINS_PER_SIDE = 20  # Total bins = 2 * NUM_BINS_PER_SIDE (symmetric around boundary)
MAX_DISTANCE = 1000  # pixels; None for auto-detection

# Smoothing for area plots
SMOOTH_SIGMA = 1.0  # Gaussian smoothing sigma; 0 for no smoothing

# Permutation test settings
N_PERMUTATIONS = 100  # Number of permutations for null hypothesis
SIG_THRESHOLD = 0.05  # p-value threshold
SIG_MARKER = '*'  # Marker for significant bins

# Verbose output
VERBOSE = True

print("Configuration loaded:")
print(f" Input file: {UNIFIED_INPUT_FILE}")
print(f" Output directory: {OUTPUT_DIR}")
print(f" Distance metric: {DISTANCE_COLUMN} (bed={DISTANCE_BED})")
print(f" Phenotype column: {PHENOTYPE_COLUMN}")
print(f" Patient grouping: {PATIENT_GROUPING}")
print(f" Comparison variables: {COMPARISON_VARS}")
print(f" Figure DPI: {FIGURE_DPI}")

## Data Loading and Preprocessing

Load the unified CSV and prepare data for analysis. Verify required columns and auto-detect comparison levels.

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING AND PREPROCESSING
# =============================================================================
print("Step 1: Loading and preprocessing data...\n")

# Load data
df = pd.read_csv(UNIFIED_INPUT_FILE, low_memory=False)
print(f"Loaded data shape: {df.shape}")
print(f"Total cells: {len(df):,}")
print(f"Total samples (batch_id): {df['batch_id'].nunique()}")
print(f"Total tissue ROIs: {df['tissue_roi'].nunique()}")

# Verify required columns
required_cols = ['batch_id', 'tissue_roi', 'tumor_bed', 'distance_tumor', 'distance_epithelia', PHENOTYPE_COLUMN]
missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

# Check which optional columns are available
optional_cols = ['patient', 'arm', 'response', 'timepoint', 'corrected', 'phenotype_ml_mr']
available_optional = [col for col in optional_cols if col in df.columns]
print(f"Available optional columns: {available_optional}")

# Filter to corrected cells if available
if 'corrected' in df.columns:
    df = df[df['corrected'] == 1].copy()
    print(f"Filtered to corrected cells: {len(df):,}")

# Auto-detect comparison levels if not specified
if COMPARISON_LEVELS is None:
    COMPARISON_LEVELS = {}
    for var in COMPARISON_VARS:
        if var in df.columns:
            COMPARISON_LEVELS[var] = sorted([v for v in df[var].unique() if pd.notna(v)])
    if VERBOSE:
        print(f"Auto-detected comparison levels: {COMPARISON_LEVELS}")

# Normalize patient selection
if PATIENT_SELECTION == 'all':
    if 'patient' in df.columns:
        patient_list = sorted(df['patient'].unique())
    else:
        patient_list = [None]
        print("Warning: 'patient' column not found; treating as single cohort")
else:
    patient_list = sorted(PATIENT_SELECTION if isinstance(PATIENT_SELECTION, list) else [PATIENT_SELECTION])

if VERBOSE:
    print(f"Patient list: {patient_list}")

# Set up patient dimensions
if patient_list != [None]:
    if PATIENT_GROUPING == 'split':
        patient_dims = [[p] for p in patient_list]
    else:
        patient_dims = [patient_list]
else:
    patient_dims = [[None]]

if VERBOSE:
    print(f"Patient dimensions for plotting: {patient_dims}")

print(f"\nData loading complete (RAW DATA - NO BED FILTERING YET)")
print(f"Ready for dynamic filtering in Step 2 cell")